In [101]:
import pandas as pd
import numpy as np
from pathlib import Path

In [102]:
DATABASE = Path.cwd().parent / "app" / "data" / "cleaned" / "data_sales_cleaned.parquet"
pd.set_option('display.max_columns', None)
df = pd.read_parquet(DATABASE)
df

,order_id,order_date,ship_date,ship_mode,customer_name,segment,state,country,market,region,product_id,category,sub_category,product_name,sales,quantity,discount,profit,shipping_cost,order_priority,year,unit_price,profit_margin
0,AG-2011-2040,2011-01-01,2011-06-01,Standard Class,Toby Braunhardt,Consumer,Constantine,Algeria,Africa,Africa,OFF-TEN-10000025,Office Supplies,Storage,"Tenex Lockers, Blue",4080000.0,2,0.0,1061400.0,354600.0,Medium,2011,2040000.0,0.26
1,IN-2011-47883,2011-01-01,2011-08-01,Standard Class,Joseph Holt,Consumer,New South Wales,Australia,APAC,Oceania,OFF-SU-10000618,Office Supplies,Supplies,"Acme Trimmer, High Speed",1200000.0,3,0.1,360360.0,97200.0,Medium,2011,400000.0,0.30
2,HU-2011-1220,2011-01-01,2011-05-01,Second Class,Annie Thurman,Consumer,Budapest,Hungary,EMEA,EMEA,OFF-TEN-10001585,Office Supplies,Storage,"Tenex Box, Single Width",660000.0,4,0.0,296400.0,81700.0,High,2011,165000.0,0.45
3,IT-2011-3647632,2011-01-01,2011-05-01,Second Class,Eugene Moren,Home Office,Stockholm,Sweden,EU,North,OFF-PA-10001492,Office Supplies,Paper,"Enermax Note Cards, Premium",450000.0,3,0.5,-260550.0,48200.0,High,2011,150000.0,-0.58
4,CA-2011-1510,2011-02-01,2011-06-01,Standard Class,Magdelene Morse,Consumer,Ontario,Canada,Canada,Canada,TEC-OKI-10002750,Technology,Machines,"Okidata Inkjet, Wireless",3140000.0,1,0.0,31200.0,241000.0,Medium,2011,3140000.0,0.01
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
25030,CA-2014-115427,2014-12-12,2015-04-01,Standard Class,Erica Bern,Corporate,California,United States,US,West,OFF-BI-10004632,Office Supplies,Binders,GBC Binding covers,210000.0,2,0.2,64750.0,20600.0,Medium,2014,105000.0,0.31
25031,UP-2014-4410,2014-12-12,2015-04-01,Standard Class,Guy Thornton,Consumer,Zaporizhzhya,Ukraine,EMEA,EMEA,OFF-AVE-10003558,Office Supplies,Labels,"Avery Round Labels, Alphabetical",280000.0,4,0.0,61200.0,17000.0,Medium,2014,70000.0,0.22
25032,MX-2014-108574,2014-12-12,2015-04-01,Standard Class,Julia Barnett,Home Office,Tamaulipas,Mexico,LATAM,North,OFF-LA-10004969,Office Supplies,Labels,"Novimex Legal Exhibit Labels, Adjustable",170000.0,3,0.0,6600.0,13200.0,Medium,2014,56700.0,0.04
25033,MO-2014-2560,2014-12-12,2015-05-01,Standard Class,Liz Preis,Consumer,Souss-Massa-Draâ,Morocco,Africa,Africa,OFF-WIL-10001069,Office Supplies,Binders,"Wilson Jones Hole Reinforcements, Clear",40000.0,1,0.0,4200.0,4900.0,Medium,2014,40000.0,0.10


In [103]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25035 entries, 0 to 25034
Data columns (total 23 columns):
 #   Column          Non-Null Count  Dtype         
---  ------          --------------  -----         
 0   order_id        25035 non-null  object        
 1   order_date      25035 non-null  datetime64[ns]
 2   ship_date       25035 non-null  datetime64[ns]
 3   ship_mode       25035 non-null  object        
 4   customer_name   25035 non-null  object        
 5   segment         25035 non-null  object        
 6   state           25035 non-null  object        
 7   country         25035 non-null  object        
 8   market          25035 non-null  object        
 9   region          25035 non-null  object        
 10  product_id      25035 non-null  object        
 11  category        25035 non-null  object        
 12  sub_category    25035 non-null  object        
 13  product_name    25035 non-null  object        
 14  sales           25035 non-null  float64       
 15  qu

In [104]:
df['customer_name'].value_counts()

customer_name
Frank Olsen         46
Laura Armstrong     45
Bart Watters        45
Kristen Hastings    44
Eric Murdock        44
                    ..
Gary Mitchum        20
Nicole Brennan      17
Sarah Bern          17
Michael Oakman      15
Darren Budd         14
Name: count, Length: 795, dtype: int64

## Create feature for new column as monthly_orders

In [105]:
# Feature engineering to extract the month from the order_date column
df['month'] = df['order_date'].dt.to_period('M')

# Assume df_monthly_sales equal to monthly order
df_monthly_customer = (
    df.groupby(['month', 'customer_name'])
    .agg(order_count=('sales', 'count'),
         total_sales=('sales', 'sum')).reset_index()
)
df_monthly_customer

,month,customer_name,order_count,total_sales
0,2011-01,Aaron Smayling,1,4970000.00
1,2011-01,Adam Bellavance,1,2257500.00
2,2011-01,Adam Hart,1,460000.00
3,2011-01,Allen Armold,1,590000.00
4,2011-01,Andrew Allen,1,1120000.00
...,...,...,...,...
10997,2014-12,Xylona Preis,3,10560000.00
10998,2014-12,Yana Sorensen,7,20064666.67
10999,2014-12,Yoseph Carroll,4,10750000.00
11000,2014-12,Zuschuss Carroll,14,26078000.00


In [106]:
import os
from dotenv import load_dotenv

ENV_PATH = Path.cwd().parent / ".env"
load_dotenv(ENV_PATH)

HUGGINGFACE_API_KEY = os.getenv("HUGGINGFACE_API_KEY")

In [107]:
# Define LLM model
from langchain_huggingface import HuggingFaceEndpoint

print("🔍 Initializing LLM model...")
model_name = "meta-llama/Llama-3.1-8B-Instruct"
api_token = HUGGINGFACE_API_KEY

if not api_token:
    print("❌ HUGGINGFACE_API_KEY is required for LLM initialization.")
    raise ValueError("HUGGINGFACE_API_KEY is required for LLM initialization.")

# Using external hosted endpoint inference wrapper
llm = HuggingFaceEndpoint(
repo_id=model_name,
task="text-generation",
max_new_tokens=512,
temperature=0.3,
huggingfacehub_api_token=api_token,
timeout=30
)

🔍 Initializing LLM model...


In [108]:
from typing import Dict
import pandas as pd

class AbuseInvestigationAgent:
    def __init__(self, llm_model=None, historical_df: pd.DataFrame = None):
        self.llm = llm_model
        self.historical_data = historical_df

    def _get_historical_context(self, input_data: Dict) -> str:
        """
        Extracts daily historical transaction frequency and sales volume 
        for a specific customer on a specific day.
        """
        customer = input_data.get('customer_name', 'Unknown')
        target_date_str = input_data.get('current_order_date') 
        
        # Default fallbacks if no history exists for this specific day
        order_count = 0
        total_sales = 0.0
        
        if self.historical_data is not None and customer != 'Unknown' and target_date_str:
            try:
                # Convert the input date string to a Pandas Timestamp
                timestamp = pd.to_datetime(target_date_str)
                
                # Extract the matching period types used in your groupby
                target_month = timestamp.to_period('M')
                target_day = int(timestamp.day)
                
                # Query matching row for Month, Day, and Customer
                match = self.historical_data[
                    (self.historical_data['month'] == target_month) & 
                    (self.historical_data['day'] == target_day) & 
                    (self.historical_data['customer_name'] == customer)
                ]
                
                if not match.empty:
                    order_count = int(match['order_count'].values[0])
                    total_sales = float(match['total_sales'].values[0])
                else:
                    print("[DEBUG] No row matched the given criteria in historical_df.")

            except Exception as e:
                print(f"[DEBUG] Error caught during context lookup: {str(e)}")

        # Formatting a security-focused context string for the LLM
        return (
            f"Daily Velocity Alert Metrics:\n"
            f"- Customer Target: {customer}\n"
            f"- Evaluation Date: {target_date_str}\n"
            f"- Orders placed ON THIS DAY: {order_count}\n"
            f"- Total volume spent ON THIS DAY: ${total_sales:,.2f}"
        )

In [111]:
# 1. Feature engineering and granular velocity tracking
df['month'] = df['order_date'].dt.to_period('M')
df['day'] = df['order_date'].dt.day

df_monthly_customer = (
    df.groupby(['month', 'day', 'customer_name'])
    .agg(order_count=('sales', 'count'),
         total_sales=('sales', 'sum')).reset_index()
)

# 2. Instantiate agent with your newly engineered granular data
agent = AbuseInvestigationAgent(llm_model=llm, historical_df=df_monthly_customer)

test_input = {
    "customer_name": "Yana Sorensen",
    "current_order_date": "2014-12-12" 
}

context_string = agent._get_historical_context(test_input)
print(context_string)

Daily Velocity Alert Metrics:
- Customer Target: Yana Sorensen
- Evaluation Date: 2014-12-12
- Orders placed ON THIS DAY: 2
- Total volume spent ON THIS DAY: $5,176,666.67
